# Option Pricing via Fourier COS Method

This notebook implements the Fourier COS method for pricing European options under the Geometric Brownian Motion model. The approach leverages Fourier-cosine expansions to approximate the risk-neutral valuation integral efficiently.

Instead of solving a PDE, we compute option prices directly from the characteristic function of the underlying process, leading to fast and highly accurate numerical results.

We consider:

- Fourier cosine expansions of the conditional density function.
- Analytical expressions for payoff coefficients.
- Efficient truncation of the integration domain.

The focus is on computational efficiency and convergence, highlighting the advantages of spectral methods over grid-based approaches.


### Structure

1. **Problem formulation**
   - Log-price transformation of the underlying process.
   - Risk-neutral pricing as an expectation integral.

2. **COS method derivation**
   - Fourier cosine series approximation.
   - Decomposition into density and payoff coefficients.

3. **Coefficient construction**
   - Implementation of analytical payoff coefficients.
   - Use of the characteristic function of GBM.

4. **Numerical implementation**
   - Domain truncation and parameter selection.
   - Assembly of the COS pricing formula.

5. **Experiments and validation**
   - Pricing European call options for given market parameters.
   - Comparison with Black–Scholes analytical prices.

6. **Convergence analysis**
   - Error behavior as the number of cosine terms increases.
   - Trade-off between accuracy and computational cost.

# 1. Problem Formulation

Let $S_t$ follow Geometric Brownian Motion under the risk-neutral measure $\mathbb{Q}$. We introduce the following log-return transformation
$$
x = \log\!\left(\frac{S_0}{K}\right), \qquad y = \log\!\left(\frac{S_T}{K}\right)
$$
so that $x$ is observed today and $y$ is the random log-price at maturity $T$. Under GBM, $y$ is normally distributed (applying Ito's lemma for $f(S_t)=\log S_t$ and considering BM features), i.e.
$$
y \sim \mathcal{N}\!\left(x + \left(r - \tfrac{1}{2}\sigma^2\right)T,\; \sigma^2 T\right)
$$


The value of a European option at time $0$ is the discounted expectation of the payoff under $\mathbb{Q}$:

$$
V(x, t) = \mathbb{E}_{\mathbb{Q}}[ e^{-rT} g(y) \mid x] = e^{-rT} \int_{\mathbb{R}} g(y)\, f(y \mid x)\, dy
$$
where
- $g(y)$ is the payoff function expressed in the log-price variable $y$,
- $f(y \mid x)$ is the conditional density of $y$ given $x$ (iow, pdf of future log-price $y$ given today's observed state $x$).

For a European call with strike $K$ and initial stock $S_0 = K e^x$, the payoff in terms of $y$ is:

$$
g(y) = \max(K e^y - K,\; 0) = K \max(e^y - 1,\; 0)
$$

The key challenge is evaluating this integral efficiently. Rather than simulating paths or solving a PDE, the COS method exploits the fact that the characteristic function of $y - x$ is known analytically under GBM and is given by 
$$
\varphi_{\text{GBM}}(u) = \exp\!\left[iu\!\left(r - \tfrac{1}{2}\sigma^2\right)T - \tfrac{1}{2}\sigma^2 T u^2\right]
$$
for $u \in \mathbb{R}$

In [ ]:
# import libraries
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

# parameters
r = 0.04       # risk-free rate
sigma = 0.30   # volatility
K = 110.0      # strike
T = 1.0        # maturity (in years)
S0 = 100       # initial stock price

def bs_call_option(S0, K, r, sigma, T):
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S0 * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

# 2. COS Method Derivation

The integration domain $\mathbb{R}$ is truncated to a finite interval $[a, b]$ chosen large enough that the density $f(y \mid x)$ is negligible outside it. Then
$$
V(x, t) \approx e^{-rT} \int_a^b g(y)\, f(y \mid x)\, dy
$$
but to model density $f(x \mid y)$ is either messy or unknown. That's why we choose to proceed with the implementation of the characteristic function instead which is known and easy to be calculated.  Having truncated now the domain to $[a, b]$, we expand the density in a Fourier cosine basis
$$
f(y \mid x) \approx \frac{2}{b-a} \sum_{k=0}^{N-1}{}' F_k(x)\, \cos\!\left(k\pi \frac{y-a}{b-a}\right)
$$
where $\sum'$ means the $k=0$ term is taken with weight $\tfrac{1}{2}$. The coefficients $F_k$ are recovered from the characteristic function $\varphi(u)$ evaluated at $u = k\pi/(b-a)$ as follows
$$
F_k(x) = \operatorname{Re}\!\left[\varphi\!\left(\frac{k\pi}{b-a}\right) e^{ik\pi (x-a)/(b-a)}\right]
$$

Substituting into the pricing integral $V = e^{-rT}\int_a^b g(y) f(y|x)\,dy$ and defining the payoff coefficients

$$
G_k = \frac{2}{b-a}\int_a^b g(y)\cos\!\left(k\pi \frac{y-a}{b-a}\right)dy
$$

yields the COS pricing formula

$$
\boxed{V(x, T) \approx e^{-rT}\,\frac{b-a}{2} \sum_{k=0}^{N-1}{}' F_k(x)\, G_k}
$$

Both $F_k$ and $G_k$ are computed analytically and no numerical integration required.



In [ ]:
# characteristic function of log-returns under GBM
def char_function(u, r, sigma, T):
    return np.exp(1j * u * (r - 0.5 * sigma**2) * T - 0.5 * sigma**2 * T * u**2)

# truncation domain [a, b] for Fourier cosine expansion
def truncation_domain(S0, K, r, sigma, T, L=12):
    x = np.log(S0 / K)
    mu = x + r * T
    width = L * np.sqrt(sigma**2 * T)
    a = mu - width
    b = mu + width
    return a, b

# density coefficients via Fourier cosine expansion
def Fk(x, a, b, r, sigma, T, N):
    k = np.arange(N)
    u = k * np.pi / (b - a)
    phi = char_function(u, r, sigma, T)
    fk = np.real(phi * np.exp(1j * k * np.pi * (x - a) / (b - a)))
    return (2 / (b - a)) * fk

# 3. Coefficient Construction

For a call option $g(y) = K\max(e^y - 1, 0)$, the integration range reduces to $[0, b]$ (since the payoff is zero for $y < 0$, i.e. $S_T < K$)

$$
G_k = \frac{2}{b-a} \int_0^b K(e^y - 1)\cos\!\left(k\pi\frac{y-a}{b-a}\right)dy
= \frac{2K}{b-a}\bigl[\chi_k(0,b) - \psi_k(0,b)\bigr]
$$

where the auxiliary integrals are defined analytically as

$$
\chi_k(c,d) = \frac{1}{1+\left(\frac{k\pi}{b-a}\right)^2}
\left[
\cos\!\left(k\pi\frac{d-a}{b-a}\right)e^d
- \cos\!\left(k\pi\frac{c-a}{b-a}\right)e^c
+ \frac{k\pi}{b-a}\sin\!\left(k\pi\frac{d-a}{b-a}\right)e^d
- \frac{k\pi}{b-a}\sin\!\left(k\pi\frac{c-a}{b-a}\right)e^c
\right]
$$

$$
\psi_k(c,d) = \begin{cases}
\frac{b-a}{k\pi}\left[\sin\!\left(k\pi\frac{d-a}{b-a}\right) - \sin\!\left(k\pi\frac{c-a}{b-a}\right)\right] & k \neq 0 \\[6pt]
d - c & k = 0
\end{cases}
$$

These are purely analytical and no numerical quadrature is needed. This is a central efficiency advantage of the COS method.

In [ ]:
# chi and psi functions for payoff coefficients G_k
def chi_k(c, d, a, b, k):
    kpi_ba = k * np.pi / (b - a)
    denom = 1.0 + kpi_ba**2

    cos_d = np.cos(kpi_ba * (d - a))
    cos_c = np.cos(kpi_ba * (c - a))
    sin_d = np.sin(kpi_ba * (d - a))
    sin_c = np.sin(kpi_ba * (c - a))

    return (cos_d * np.exp(d) - cos_c * np.exp(c) + kpi_ba * (sin_d * np.exp(d) - sin_c * np.exp(c))) / denom

def psi_k(c, d, a, b, k):
    result = np.zeros_like(k, dtype=float)
    result[k == 0] = d - c
    nz = k != 0
    kpi_ba = k[nz] * np.pi / (b - a)
    result[nz] = (b - a) / (k[nz] * np.pi) * (np.sin(kpi_ba * (d - a)) - np.sin(kpi_ba * (c - a)))
    return result


# payoff coefficients G_k (European call)
def Gk_call(a, b, K, N):
    k = np.arange(N, dtype=float)
    c, d = 0.0, b  # payoff is nonzero only for y in [0, b]
    chi = chi_k(c, d, a, b, k)
    psi = psi_k(c, d, a, b, k)
    return (2.0 * K / (b - a)) * (chi - psi)

# 4. Numerical Implementation

The integration domain $[a, b]$ is chosen to cover $L$ standard deviations of $y$ around its mean $\mu = x + rT$:

$$
a = x + rT - L\sigma\sqrt{T}, \qquad b = x + rT + L\sigma\sqrt{T}
$$

The assignment specifies $L = 12$, which ensures the truncation error is negligible. Given $N$ cosine terms, the price is computed as:

1. Form the index vector $k = 0, 1, \ldots, N-1$
2. Evaluate $F_k(x)$ from the characteristic function
3. Evaluate $G_k$ analytically via $\chi_k$, $\psi_k$
4. Apply the half-weight to $k=0$ and sum

$$
V \approx e^{-rT}\,\frac{b-a}{2}\left[\tfrac{1}{2}F_0 G_0 + \sum_{k=1}^{N-1} F_k G_k\right]
$$


In [ ]:
def cos_pricing(S0, K, r, sigma, T, N=64, L=6):
    x   = np.log(S0 / K)
    a, b = truncation_domain(S0, K, r, sigma, T, L=L)

    # compute coefficients
    fk  = Fk(x, a, b, r, sigma, T, N)
    gk  = Gk_call(a, b, K, N)

    fk[0] *= 0.5 # half-weight on k=0 term
    price = np.exp(-r * T) * (b - a) / 2 * np.dot(fk, gk)
    return price

p_cos = cos_pricing(S0, K, r, sigma, T, N=64)
print(f"COS price: {p_cos:.6f}")

# 5. Experiments and Validation

We price a European call for three moneyness levels using $N = 64$ cosine terms and compare against the Black–Scholes closed-form price. Results confirm that the COS method achieves errors well below $10^{-6}$ for all three cases with very few terms (see table below - that happens only with $N=20$!).


In [ ]:
# COS price vs Black–Scholes price for different moneyness cases
S0_cases = {"ITM": 100.0, "ATM": 110.0, "OTM": 120.0}

# plotting 
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=False)

N_range = np.arange(2, 65, 2)

for ax, (label, S0) in zip(axes, S0_cases.items()):
    p_bs  = bs_call_option(S0, K, r, sigma, T)
    prices = [cos_pricing(S0, K, r, sigma, T, N=n) for n in N_range]

    ax.plot(N_range, prices, color="steelblue", marker="o", markersize=3)
    ax.axhline(p_bs, color="tomato", linestyle="--", linewidth=1.2, label=f"BS = {p_bs:.4f}")
    ax.set_title(f"{label}  ($S_0={S0}$)")
    ax.set_xlabel("$N$ cosine terms")
    ax.set_ylabel("Option price")
    ax.legend(fontsize=8)

plt.suptitle("COS price vs Black–Scholes - European call", y=1.02)
plt.tight_layout()
plt.show()

N = 20
print(f"{'Case':>6}  {'COS price':>14}  {'BS price':>14}  {'Error':>14}")
print("-" * 54)
for label, S0 in S0_cases.items():
    p_cos = cos_pricing(S0, K, r, sigma, T, N=N)
    p_bs  = bs_call_option(S0, K, r, sigma, T)
    err = abs(p_cos - p_bs)
    print(f"{label:>6}  {p_cos:14.6f}  {p_bs:14.6f}  {err:14.2e}")

We see that we very few cosine terms ($N < 10$) - as it happens when using simulations - the basis is too coarse to represent the payoff function. The call payoff has a kink at $y=0$ where $S_T = K$ which requires many terms to approximate.

# 6. Convergence Analysis

We track the absolute error $|V_{\text{COS}} - V_{\text{BS}}|$ as $N$ increases from $1$ to $200$. The COS method converges exponentially in $N$ for smooth densities (such as the Gaussian under GBM) - $\mathcal{O}(e^{-\alpha N})$ for some $\alpha >0$. This is the key advantage over grid-based PDE methods, which achieve at best second-order (algebraic) convergence in the number of grid points.

In [ ]:
S0 = S0_cases["ITM"]
p_bs = bs_call_option(S0, K, r, sigma, T)
N_vals = np.arange(1, 201)
errors = [abs(cos_pricing(S0, K, r, sigma, T, N=n) - p_bs) for n in N_vals]

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(N_vals, errors, linewidth=0.8, color="steelblue")
ax.set_xlabel("$N$ cosine terms")
ax.set_ylabel("Absolute error  V_COS - V_BS")
ax.set_title("Convergence of COS method - ITM European call")
plt.tight_layout()
plt.show()

The error drops approximately 9 orders of magnitude in under 25 terms, hitting floating-point machine precision (~$10^{-8}$) well before $N = 30$. A finite difference scheme achieving $\mathcal{O}(N^{-2})$ from $\Delta X = ( X_{\text{max}} - X_{\text{min}})/(N+1)$, and it would require more than 300 hundreds of grid points - as we saw in the related notebook - to reach comparable accuracy, at a significantly higher computational cost.